# 03 — Gold Layer

**Tujuan:** Membuat tabel agregasi bisnis dari Silver Layer untuk analisis nutrisi global dan distribusi NOVA/Eco-Score.

**Input:** `/home/jovyan/work/data/silver/food_clean` (Delta Lake)  
**Output:**
- `/home/jovyan/work/data/gold/nutrition_by_category`
- `/home/jovyan/work/data/gold/nutrition_by_country`
- `/home/jovyan/work/data/gold/nova_distribution`
- `/home/jovyan/work/data/gold/eco_distribution`

**Pipeline:**
1. Setup SparkSession
2. Baca Silver Layer + drop kolom NULL
3. Agregasi nutrisi per kategori
4. Agregasi nutrisi per negara
5. Distribusi NOVA group & Eco-Score global
6. Simpan ke Delta Lake
7. Validasi hasil

## 1. Setup SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[2]") \
    .appName("03-gold-layer") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.maxResultSize", "1g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

print("Spark version:", spark.version)
print("Mode:", spark.sparkContext.master)

Spark version: 3.5.1
Mode: local[2]


## 2. Baca Silver Layer

> **Catatan:** Kolom `created_dt` dan `last_modified_dt` di-drop karena berisi NULL semua (hasil ALTER TABLE yang tidak berhasil diisi akibat keterbatasan RAM).

In [2]:
# Aktifkan column mapping + drop kolom NULL
spark.sql("""
    ALTER TABLE delta.`/home/jovyan/work/data/silver/food_clean`
    SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    )
""")

spark.sql("""
    ALTER TABLE delta.`/home/jovyan/work/data/silver/food_clean`
    DROP COLUMN created_dt
""")
spark.sql("""
    ALTER TABLE delta.`/home/jovyan/work/data/silver/food_clean`
    DROP COLUMN last_modified_dt
""")

df_silver = spark.read.format("delta").load("/home/jovyan/work/data/silver/food_clean")

print("Jumlah baris:", df_silver.count())
print("Jumlah kolom:", len(df_silver.columns))

Jumlah baris: 1119410
Jumlah kolom: 121


## 3. Agregasi Nutrisi per Kategori

In [3]:
from pyspark.sql.functions import avg, count, round as spark_round

df_gold_category = df_silver.groupBy("categories") \
    .agg(
        count("*").alias("jumlah_produk"),
        spark_round(avg("energy_kcal_100g"), 2).alias("avg_energy_kcal"),
        spark_round(avg("fat_100g"), 2).alias("avg_fat_g"),
        spark_round(avg("sugars_100g"), 2).alias("avg_sugars_g"),
        spark_round(avg("proteins_100g"), 2).alias("avg_proteins_g"),
        spark_round(avg("salt_100g"), 2).alias("avg_salt_g")
    ) \
    .filter("jumlah_produk >= 10") \
    .orderBy("jumlah_produk", ascending=False)

print("Agregasi per kategori selesai")
print("Jumlah kategori:", df_gold_category.count())
df_gold_category.show(5, truncate=50)

Agregasi per kategori selesai
Jumlah kategori: 10172
+-------------------------------------+-------------+---------------+---------+------------+--------------+----------+
|                           categories|jumlah_produk|avg_energy_kcal|avg_fat_g|avg_sugars_g|avg_proteins_g|avg_salt_g|
+-------------------------------------+-------------+---------------+---------+------------+--------------+----------+
|                                 NULL|        94777|         267.11|    10.27|       12.72|          6.96|      1.54|
|                            undefined|        33744|         273.61|    42.07|        40.9|         17.93|      0.16|
|                               Snacks|        31011|          451.7|    24.89|       18.34|         12.01|      1.19|
|                                     |        23234|         290.69|    12.23|       16.85|          8.06|      1.31|
|Snacks, Sweet snacks, Confectioneries|        13376|         374.01|      7.1|       56.57|          2.57|      0

## 4. Agregasi Nutrisi per Negara

In [4]:
from pyspark.sql.functions import explode, col

df_gold_country = df_silver \
    .withColumn("country", explode(col("countries_tags"))) \
    .groupBy("country") \
    .agg(
        count("*").alias("jumlah_produk"),
        spark_round(avg("energy_kcal_100g"), 2).alias("avg_energy_kcal"),
        spark_round(avg("fat_100g"), 2).alias("avg_fat_g"),
        spark_round(avg("sugars_100g"), 2).alias("avg_sugars_g"),
        spark_round(avg("proteins_100g"), 2).alias("avg_proteins_g"),
        spark_round(avg("salt_100g"), 2).alias("avg_salt_g")
    ) \
    .filter("jumlah_produk >= 10") \
    .orderBy("jumlah_produk", ascending=False)

print("Agregasi per negara selesai")
print("Jumlah negara:", df_gold_country.count())
df_gold_country.show(5, truncate=40)

Agregasi per negara selesai
Jumlah negara: 201
+-----------------+-------------+---------------+---------+------------+--------------+----------+
|          country|jumlah_produk|avg_energy_kcal|avg_fat_g|avg_sugars_g|avg_proteins_g|avg_salt_g|
+-----------------+-------------+---------------+---------+------------+--------------+----------+
| en:united-states|       385666|         271.76|    14.87|       17.78|          8.14|       1.6|
|        en:france|       331511|         277.23|    14.18|       13.25|          7.52|      1.07|
|       en:germany|       109359|         267.68|    13.43|       12.51|          7.53|      1.23|
|en:united-kingdom|        55369|         269.83|    12.34|       11.83|          7.39|      1.12|
|         en:spain|        52495|         303.35|    17.79|       10.93|          7.04|      1.15|
+-----------------+-------------+---------------+---------+------------+--------------+----------+
only showing top 5 rows


## 5. Distribusi NOVA Group & Eco-Score Global

In [5]:
df_nova_dist = df_silver.groupBy("nova_group") \
    .agg(count("*").alias("jumlah_produk")) \
    .orderBy("nova_group")

print("=== Distribusi NOVA Group ===")
df_nova_dist.show()

df_eco_dist = df_silver.groupBy("environmental_score_grade") \
    .agg(count("*").alias("jumlah_produk")) \
    .orderBy("environmental_score_grade")

print("=== Distribusi Eco-Score ===")
df_eco_dist.show()

=== Distribusi NOVA Group ===
+----------+-------------+
|nova_group|jumlah_produk|
+----------+-------------+
|         1|       133351|
|         2|        63691|
|         3|       207395|
|         4|       714973|
+----------+-------------+

=== Distribusi Eco-Score ===
+-------------------------+-------------+
|environmental_score_grade|jumlah_produk|
+-------------------------+-------------+
|                     NULL|       269985|
|                        a|        68003|
|                   a-plus|        41634|
|                        b|       108827|
|                        c|        70589|
|                        d|        77636|
|                        e|        58638|
|                        f|        23998|
|           not-applicable|        23926|
|                  unknown|       376174|
+-------------------------+-------------+


## 6. Simpan ke Delta Lake

In [6]:
import time

start = time.time()

df_gold_category.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/home/jovyan/work/data/gold/nutrition_by_category")

df_gold_country.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/home/jovyan/work/data/gold/nutrition_by_country")

df_nova_dist.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/home/jovyan/work/data/gold/nova_distribution")

df_eco_dist.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/home/jovyan/work/data/gold/eco_distribution")

elapsed = time.time() - start
print(f"Gold Layer tersimpan. Waktu: {elapsed:.1f} detik")

Gold Layer tersimpan. Waktu: 12.4 detik


## 7. Validasi Hasil

In [7]:
df_cat = spark.read.format("delta").load("/home/jovyan/work/data/gold/nutrition_by_category")
df_ctr = spark.read.format("delta").load("/home/jovyan/work/data/gold/nutrition_by_country")
df_nova = spark.read.format("delta").load("/home/jovyan/work/data/gold/nova_distribution")
df_eco = spark.read.format("delta").load("/home/jovyan/work/data/gold/eco_distribution")

print("Tabel agregasi per kategori:", df_cat.count(), "kategori")
print("Tabel agregasi per negara:", df_ctr.count(), "negara")
print("Distribusi NOVA:", df_nova.count(), "group")
print("Distribusi Eco-Score:", df_eco.count(), "grade")
print("\n✅ Gold Layer selesai!")

Tabel agregasi per kategori: 10172 kategori
Tabel agregasi per negara: 201 negara
Distribusi NOVA: 4 group
Distribusi Eco-Score: 10 grade

✅ Gold Layer selesai!


## 8. Catat Throughput

In [8]:
throughput = 1119410 / 12.4
print(f"Throughput Gold: {throughput:,.0f} baris/detik")

Throughput Gold: 90,275 baris/detik
